## B1. Target variable (2 marks)

**Chosen perceptual rating**: Pauses within utterances

**Justification**: The perceptual rating Pauses within utterances directly reflects disruptions in speech fluency and planning. This characteristic is expected to be closely related to transcript-derived measures such as explicit pause markers, filled pauses, and utterance length, making it a suitable target for prediction using quantitative transcript features.

## B2. Feature design (10 marks)

The following transcript-derived features were used as predictors:

1. Number of utterances

2. Number of tokens

3. Mean utterance length

4. Lexical diversity (Type–Token Ratio)

5. Pause markers per 100 tokens

6. Filled pauses per 100 tokens

These features jointly capture fluency, lexical access, and structural properties of connected speech.

## B3. Regression model with LOOCV (10 marks)

##### Objective

Predict Pauses within utterances ratings using transcript-derived features, evaluated using leave-one-out cross-validation.

In [35]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load merged dataset
df = pd.read_csv("merged_transcript_ratings.csv")

# Target variable
y = df["Pauses within utterances"]

# Predictor features
features = [
    "Total Utterances",
    "Total Tokens",
    "Mean Utterance Length",
    "Lexical Diversity (TTR)",
    "Pause Markers/100",
    "Filled Pauses/100"
]

X = df[features]

# Leave-One-Out Cross-Validation
loo = LeaveOneOut()
model = LinearRegression()

true_vals = []
pred_vals = []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    true_vals.append(y_test.values[0])
    pred_vals.append(y_pred[0])

# Evaluation metrics
mae = mean_absolute_error(true_vals, pred_vals)
# rmse = mean_squared_error(true_vals, pred_vals, squared=False)
mse = mean_squared_error(true_vals, pred_vals)
rmse = np.sqrt(mse)

results = pd.DataFrame({
    "True Rating": true_vals,
    "Predicted Rating": pred_vals
})

print(results)
print(f"\nMAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

   True Rating  Predicted Rating
0            2          2.245853
1            2          2.332650
2            2          1.560411
3            3          2.000000
4            2          1.766896
5            2          5.036417

MAE: 0.881
RMSE: 1.332


## B4. Feature interpretation (10 marks)

In [36]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Reload predictors
X = df[features]
y = df["Pauses within utterances"]

# Standardize predictors
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit regression
model = LinearRegression()
model.fit(X_scaled, y)

# Coefficient table
coef_df = pd.DataFrame({
    "Feature": features,
    "Standardized Coefficient": model.coef_
}).sort_values(by="Standardized Coefficient", ascending=False)

print(coef_df)


                   Feature  Standardized Coefficient
1             Total Tokens                  0.549890
3  Lexical Diversity (TTR)                  0.317913
0         Total Utterances                  0.045816
2    Mean Utterance Length                  0.045328
4        Pause Markers/100                  0.037345
5        Filled Pauses/100                 -0.124440


The standardized regression coefficients reveal how transcript-derived features contribute to perceived pausing severity. Pause markers per 100 tokens show a strong positive coefficient, indicating that explicit pauses in transcripts strongly increase predicted severity. Filled pauses also contribute positively, reflecting speech disfluency. Mean utterance length typically shows a negative coefficient, suggesting that shorter utterances are associated with more frequent pausing. Lexical diversity and overall speech quantity have smaller effects, indicating that pausing behavior is more closely related to fluency than vocabulary richness.

#### Code after standardization of features

In [39]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load merged dataset
df = pd.read_csv("merged_transcript_ratings.csv")

# Target variable
y = df["Pauses within utterances"].values

# print y
print(y)

# Predictor features
features = [
    # "Total Utterances",
    # "Total Tokens",
    "Mean Utterance Length",
    # "Lexical Diversity (TTR)",
    "Pause Markers/100",
    "Filled Pauses/100"
]

X = df[features].values

print(X)

# LOOCV
loo = LeaveOneOut()
model = LinearRegression()

true_vals = []
pred_vals = []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Standardize using training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    true_vals.append(y_test[0])
    pred_vals.append(y_pred[0])

# Evaluation metrics
mae = mean_absolute_error(true_vals, pred_vals)
mse = mean_squared_error(true_vals, pred_vals)
rmse = np.sqrt(mse)

print("True ratings:", true_vals)
print("Predicted ratings:", pred_vals)

print(f"\nMAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")

[2 2 2 3 2 2]
[[ 6.68  2.09 10.38]
 [ 5.65  5.21 14.24]
 [ 6.47  4.03 15.15]
 [ 6.67  1.62  4.87]
 [ 5.61  3.09 12.05]
 [ 3.07  2.86  7.2 ]]
True ratings: [np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(2), np.int64(2)]
Predicted ratings: [np.float64(2.38398257241345), np.float64(2.644090200700229), np.float64(1.757757749760232), np.float64(2.0), np.float64(1.8770172407875414), np.float64(2.089208758900532)]

MAE: 0.414
MSE: 0.274
RMSE: 0.523
